# SWG query training — in-process Colab launcher

This version avoids subprocess entirely. The experiment runs inside the Colab kernel, so all `print()` output is visible. At the end it also reads the saved Drive JSON and prints the key panels again.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, shutil, subprocess, runpy, json, pathlib
REPO='/content/Sparsewalker'
BRANCH='agent/walker-swg-query-training'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
subprocess.run([sys.executable,'-m','pip','install','-q','faiss-cpu'],check=True)
SRC=f'{REPO}/src'
EXP=f'{REPO}/experiments'
sys.path.insert(0,SRC)
sys.path.insert(0,EXP)
os.chdir(REPO)

OUT='/content/drive/MyDrive/sparsewalker_swg_query/result.json'
sys.argv=[
    'run_swg_query_training.py',
    '--seed','42',
    '--epochs','5',
    '--train-cap','300000',
    '--per-bucket','128',
    '--hnsw-m','8',
    '--ef-construction','160',
    '--output',OUT,
]
print('INPROCESS RUN START', flush=True)
runpy.run_path(f'{REPO}/experiments/run_swg_query_training.py', run_name='__main__')
print('INPROCESS RUN END', flush=True)

p=pathlib.Path(OUT)
print('RESULT_FILE', str(p), 'exists=', p.exists(), 'bytes=', p.stat().st_size if p.exists() else 0, flush=True)
if not p.exists():
    raise FileNotFoundError(f'Expected result file was not written: {p}')
r=json.loads(p.read_text())
print('\nRECOVERED_PRETRAIN_PANEL', json.dumps({
    'dense_target_hit@10': r['pretrain']['dense_target_hit@10'],
    'hop4': r['pretrain']['navigation']['4'],
}, indent=2), flush=True)
for row in r['history']:
    print('RECOVERED_QUERY_TRAIN', row, flush=True)
print('\nRECOVERED_POSTTRAIN_PANEL', json.dumps({
    'dense_target_hit@10': r['posttrain']['dense_target_hit@10'],
    'hop4': r['posttrain']['navigation']['4'],
    'oracle_next_hop4': r['oracle_next']['4'],
}, indent=2), flush=True)
pre=r['pretrain']['navigation']['4']['next_item_concept_seen_rate']
post=r['posttrain']['navigation']['4']['next_item_concept_seen_rate']
print('RECOVERED_DECISION', {
    'pre_next_seen_h4': pre,
    'post_next_seen_h4': post,
    'absolute_gain': post-pre,
    'dense_target_hit@10': r['posttrain']['dense_target_hit@10'],
    'oracle_next_h4': r['oracle_next']['4']['any_target_hit_rate'],
}, flush=True)
